In [2]:
import time
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm

from src.parameters import MAX_PRECIP_MM, MIN_TEMP_C, MAX_TEMP_C

tqdm.pandas()

/Users/Patron/Documents/cs524/.gamspy_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
sundays_df = pd.read_csv('data/processed/sundays_2026.csv')
circuits_df = pd.read_csv('data/processed/circuits.csv')
sundays_df.head()

,race_date,week_num,month
0,2026-01-04,1,1
1,2026-01-11,2,1
2,2026-01-18,3,1
3,2026-01-25,4,1
4,2026-02-01,5,2


In [4]:
# Get season start and end dates

start_date = sundays_df[sundays_df['month'] == 3]['race_date'].values[0]
end_date = sundays_df[sundays_df['month'] == 12]['race_date'].values[0]

start_week = sundays_df[sundays_df['month'] == 3]['week_num'].values[0]

print(f"Season start date: {start_date} (week: {start_week}), end date: {end_date}")


Season start date: 2026-03-01 (week: 9), end date: 2026-12-06


In [5]:
mask = (sundays_df['race_date'] >= start_date) & (sundays_df['race_date'] <= end_date)

race_weekends_df = sundays_df.loc[mask].reset_index(drop=True)
race_weekends_df['week_num'] = race_weekends_df['week_num'] - start_week + 1

race_weekends_df.head()

,race_date,week_num,month
0,2026-03-01,1,3
1,2026-03-08,2,3
2,2026-03-15,3,3
3,2026-03-22,4,3
4,2026-03-29,5,3


In [6]:
API_KEY = "U2333EX6VCZBN2545ACQYFPV4"
# API_KEY = "9C84VDUB86RH2E9ATA5HZAWAV"
# API_KEY = "L6P4AVRPDA2YQSCVCUZG8STPG"

def get_weather_history(row, coords, years=5):
    base_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline"
    params = {
        "key": API_KEY,
        "include": "days",
        "contentType": "json"
    }
    temps = []
    precips = []
    date = row['race_date']

    for year_offset in range(1, years+1):
        year = int(date[:4]) - year_offset
        new_date = f"{year}{date[4:]}"
        url = f"{base_url}/{coords}/{new_date}"
        response = requests.get(url, params=params)

        try:
            res = response.json()
            day_data = res['days'][0]
            temps.append(day_data['temp'])
            precips.append(day_data['precip'])

        except Exception as e:
            print(f"Error fetching data for {coords} on {new_date}: {e}")
            exit(1)
        time.sleep(1)  # To respect API rate limits

    # Return average temperature and precipitation with 2 decimal places
    return pd.Series([round(np.mean(temps), 2), round(np.mean(precips), 2)])

In [7]:
for index, circuit in circuits_df.iterrows():


    circuit_id = circuit['circuit_id']
    coords = f"{circuit['latitude']},{circuit['longitude']}"
    tobedone = []

    print(f"Processing circuit: {circuit_id} at {coords} [{index+1}/{len(circuits_df)}]")
    if circuit_id in tobedone:
        race_weekends_df[[f'{circuit_id}_avg_temp', f'{circuit_id}_avg_precip']] = race_weekends_df.progress_apply(lambda row: get_weather_history(row, coords), axis=1)

Processing circuit: BAH at 26.0316553,50.5145563 [1/24]
Processing circuit: SAU at 21.5504432,39.1742363 [2/24]
Processing circuit: AUS at -37.8142454,144.9631732 [3/24]
Processing circuit: JPN at 34.8817102,136.5836516 [4/24]
Processing circuit: CHN at 31.2312707,121.4700152 [5/24]
Processing circuit: MIA at 25.7741566,-80.1935973 [6/24]
Processing circuit: MON at 43.7402961,7.426559 [7/24]
Processing circuit: CAN at 45.5031824,-73.5698065 [8/24]
Processing circuit: ESP at 41.3825802,2.177073 [9/24]
Processing circuit: MAD at 40.416782,-3.703507 [10/24]
Processing circuit: AUT at 47.2122736,14.7857412 [11/24]
Processing circuit: GBR at 52.0917705,-1.0260194 [12/24]
Processing circuit: HUN at 47.5986636,19.2384215 [13/24]
Processing circuit: BEL at 50.3942409,5.9310335 [14/24]
Processing circuit: NED at 52.3719838,4.5302209 [15/24]
Processing circuit: MONZ at 45.6395418,9.2788304 [16/24]
Processing circuit: SIN at 1.2899175,103.8519072 [17/24]
Processing circuit: AZE at 40.3755885,49.8